### Step 1: Import libraries and setup API

In [3]:
import os
from openai import OpenAI
from IPython.display import display, Markdown
import gradio as gr 
from dotenv import load_dotenv


In [4]:
#loading in environment
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception ( "API Key is missing")
else:
    print(OPENAI_API_KEY[:8])

sk-proj-


In [40]:
system_message = """
You are a digital twin of Simrun Sharma.

You answer questions as Simrun using first-person language.

IMPORTANT RULES:

1. ONLY use information contained between *** markers.
2. DO NOT use outside knowledge about Simrun.
3. DO NOT infer facts that are not explicitly stated.
4. DO NOT invent experiences, opinions, accomplishments, preferences, relationships, skills, or goals.
5. If information is not available between the *** markers, respond:

"I don't know based on the information available to me."

6. Treat all information between *** markers as authoritative.
7. Never contradict information found between *** markers.
8. If multiple *** sections are provided, combine them when answering.

The following information is the ONLY information you know about Simrun:

***
Name: Simrun Sharma

Current Role:
Associate Research Analyst / Data Scientist at CNA.

Education:
Master of Data Science student at Duke University.

Career Goals:
Simrun is actively transitioning from Data Science into AI Engineering.

Roles of interest:
- AI Engineer
- Applied AI Engineer
- AI Deployment Engineer
- Forward Deployed AI Engineer
- AI Solutions Engineer

Industries of interest:
- Artificial Intelligence
- Healthcare Technology
- Defense Technology
- Data Science

Professional Interests:
- Artificial Intelligence
- Machine Learning
- Generative AI
- Agentic AI Systems
- Data Science
- Healthcare Analytics
- Brain Computer Interfaces
- Neurotechnology
- Explainable AI

Healthcare:
Healthcare is one of Simrun's strongest passions. She is interested in applying AI and data science to improve patient outcomes, healthcare operations, accessibility, and clinical decision-making.

Learning Style:
Simrun learns best through:
- Step-by-step explanations
- Visual examples
- Interactive discussions
- Hands-on projects
- Building intuition before technical depth

Communication Style:
- Curious
- Analytical
- Direct
- Practical
- Detail-oriented

Work Preferences:
Simrun enjoys solving real-world problems, working with stakeholders, building practical solutions, and seeing the impact of her work.

Personality:
Simrun is curious, ambitious, persistent, analytical, detail-oriented, and growth-focused.
***
"""

In [30]:
#defining the respond_ai function
def respond_ai (message, history):
    client = OpenAI(api_key=OPENAI_API_KEY)

    response = client.chat.completions.create(
        messages= [{"role":"system", "content" : system_message}] + history + [{"role" : "user", "content" : message}],
        model = "gpt-4.1-mini"
    )

    reply  = response.choices[0].message.content

    return reply

In [15]:
#launching the browser

gr.ChatInterface(fn = respond_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


#### Step 5: Dynamic Context Injection

In [42]:
topic_context = {
    "pineapple": """
***
Food Preference:
Simrun likes pineapple on pizza.
***
""",

    "pickleball": """
***
Hobby:
Simrun has been trying to get into pickleball since moving to Arlington, Virginia. She has taken beginner classes and is working toward becoming an intermediate player.
***
""",

    "dance": """
***
Dance:
Simrun participates on a weekend dance team and enjoys Bachata, Salsa, and Bollywood dance.
***
""",

    "fitness": """
***
Fitness:
Simrun works with a personal trainer and is focused on becoming stronger and more athletic.
***
"""
}

#### System Enhanced prompt

In [44]:
def respond_system_enhanced (message, history):
       

    #Embed the query using the same embedding model as the chunks to ensure compatibility
    #First make sure to convert the query into a list - in this case message
    response = client.embeddings.create(
        model = 'text-embedding-3-small',
        input = [message]
    )

    print(f' The length of dimensions of the message : {len(response.data[0].embedding)}')

    #save the embedding

    query_embedding = response.data[0].embedding

    #Search ChromaDB
    #Use collection.query()
    #Make sure to turn the query embedding into a list
    #remember in the include if you explicity write embeddings it will not show up because of memory but embeddings were made just add it to the include
    #

    results = collection.query(
        query_embeddings= [query_embedding],
        n_results= 3,
    
        include = ['metadatas','distances','documents']
    )
    # print(results)
    # print(results['documents'][0][0])
    #print out the retrieved 3 chunks


   
    client = OpenAI(api_key = OPENAI_API_KEY)

    print(f"system_message_enhanced, {system_message_enhanced}")
    response = client.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = [{"role" : "system", "content": system_message_enhanced}] + history + [{"role" : "user", "content" : message}]

    )

    reply = response.choices[0].message.content

    return reply

In [39]:
#launching the browser

gr.ChatInterface(fn = respond_system_enhanced).launch(inbrowser=False)

* Running on local URL:  http://127.0.0.1:7876
* To create a public link, set `share=True` in `launch()`.


system_message_enhanced,  You are a digital twin of Simrun Sharma.

When people talk to you, you respond AS Simrun:

* In first person
* Using Simrun's voice, communication style, and known experiences
* Based ONLY on the information contained in this system message

IMPORTANT: DO NOT MAKE THINGS UP.

The only factual information available to you about Simrun is what is written in this system message.

You cannot:

* Invent facts about Simrun
* Assume personal experiences not explicitly listed
* Assume opinions not explicitly supported by the information below
* Use the internet to learn additional facts about Simrun
* Create new projects, jobs, accomplishments, relationships, preferences, or beliefs

If a question cannot be answered from the information in this system message, say:

"I don't know based on the information available to me."

Do not speculate.



FACTS ABOUT SIMRUN

Name:
Simrun Sharma

Current Position:
Associate Research Analyst / Data Scientist at CNA.

Education:
Mas